# Deep Learning Vision Lab

Source lecture: `#7 딥러닝 비전`

This notebook follows the lecture flow with compact PyTorch examples:

1. MNIST classification with an MLP
2. LeNet-style CNN reproduction
3. Data augmentation
4. Pre-trained ResNet inference
5. Mask R-CNN instance segmentation
6. YOLO object detection


## Environment

Install packages if needed:

```bash
pip install torch torchvision torchaudio matplotlib numpy pillow ultralytics notebook
```


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn
from torchvision.transforms.functional import to_tensor

ROOT = Path('.')
DATA_DIR = ROOT / 'data'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


## 1. MNIST with a Simple MLP

This mirrors the lecture section that starts from fully connected classification on MNIST.


In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_set = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=mnist_transform)
test_set = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=mnist_transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

class MnistMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.net(x)

mlp = MnistMLP().to(DEVICE)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


In [ ]:
mlp.train()
for step, (images, labels) in enumerate(train_loader, start=1):
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)
    optimizer.zero_grad()
    logits = mlp(images)
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
    if step == 100:
        print('step', step, 'loss', float(loss))
        break


In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = mlp(images[:9].to(DEVICE)).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for ax, image, label, pred in zip(axes.flat, images[:9], labels[:9], preds):
    ax.imshow(image.squeeze(0), cmap='gray')
    ax.set_title(f'gt={label.item()} pred={pred.item()}')
    ax.axis('off')
plt.tight_layout()


## 2. LeNet-5 Style CNN

The lecture revisits LeNet as a compact convolutional network for handwritten digits.


In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(2),
            nn.Conv2d(6, 16, kernel_size=5),
            nn.Tanh(),
            nn.AvgPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 120),
            nn.Tanh(),
            nn.Linear(120, 84),
            nn.Tanh(),
            nn.Linear(84, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

lenet = LeNet5().to(DEVICE)
optimizer = torch.optim.Adam(lenet.parameters(), lr=1e-3)


In [ ]:
lenet.train()
for step, (images, labels) in enumerate(train_loader, start=1):
    images = images.to(DEVICE)
    labels = labels.to(DEVICE)
    optimizer.zero_grad()
    logits = lenet(images)
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
    if step == 100:
        print('step', step, 'loss', float(loss))
        break


## 3. Data Augmentation

The lecture emphasizes augmentation as a practical regularization tool after AlexNet.


In [ ]:
image = Image.open(DATA_DIR / 'cat.jpg').convert('RGB')
augment = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    ax.imshow(image if idx == 0 else augment(image))
    ax.set_title('original' if idx == 0 else f'aug {idx}')
    ax.axis('off')
plt.tight_layout()


## 4. Pre-trained ResNet Inference

This section corresponds to the lecture part that uses pre-trained classification backbones.


In [ ]:
weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights).to(DEVICE)
resnet.eval()
preprocess = weights.transforms()
categories = weights.meta['categories']

sample = Image.open(DATA_DIR / 'dog1.jpg').convert('RGB')
batch = preprocess(sample).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    probs = torch.softmax(resnet(batch), dim=1)[0]
values, indices = probs.topk(5)
labels = [categories[i] for i in indices.cpu().tolist()]
scores = values.cpu().tolist()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample)
axes[0].axis('off')
axes[0].set_title('input image')
axes[1].barh(range(5), scores[::-1], color='#4f8cff')
axes[1].set_yticks(range(5), labels[::-1])
axes[1].set_xlim(0.0, 1.0)
axes[1].set_title('top-5 predictions')
plt.tight_layout()


## 5. Mask R-CNN Instance Segmentation

The lecture introduces Mask R-CNN as a two-stage detector with a mask branch.


In [ ]:
weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
mask_model = maskrcnn_resnet50_fpn(weights=weights).to(DEVICE)
mask_model.eval()
categories = weights.meta['categories']

mask_image = Image.open(DATA_DIR / 'view.jpg').convert('RGB')
tensor = to_tensor(mask_image).to(DEVICE)
with torch.no_grad():
    pred = mask_model([tensor])[0]

mask_np = np.array(mask_image).astype(np.float32) / 255.0
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mask_np)
ax.axis('off')
for idx, score in enumerate(pred['scores'][:5].cpu().tolist()):
    if score < 0.7:
        continue
    box = pred['boxes'][idx].cpu().numpy()
    label_id = int(pred['labels'][idx].cpu().item())
    ax.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1], fill=False, edgecolor='lime', linewidth=2))
    ax.text(box[0], box[1] - 5, f"{categories[label_id]} {score:.2f}", color='lime')
plt.tight_layout()


## 6. YOLO Detection

The lecture closes with one-stage detectors such as YOLO. This cell uses Ultralytics YOLO for a quick demo.


In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')
result = yolo(str(DATA_DIR / 'view.jpg'), verbose=False)[0]
annotated = result.plot()[:, :, ::-1]
plt.figure(figsize=(10, 6))
plt.imshow(annotated)
plt.axis('off')
plt.title('YOLO result')
plt.show()


## Next Steps

- Increase training epochs for the MNIST models.
- Replace the sample image with your own photo for ResNet, Mask R-CNN, or YOLO.
- Extend the pre-trained classification part into transfer learning with a custom dataset.
